# Prithvi-EO 2.0 — Development & Testing

Interactive notebook for testing the Prithvi-EO 2.0-600M foundation model on shoreline
segmentation tasks.  Covers:
1. Loading & inspecting multispectral GeoTIFFs
2. Band selection & HLS-compatible normalisation
3. MAE reconstruction (cloud infill)
4. Segmentation inference
5. Comparison with existing YOLO masks
6. Batch site processing via `pipeline_adapter`

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install terratorch torch rasterio huggingface_hub scipy Pillow einops matplotlib

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch

# Add prithvi2/ to path
PRITHVI2_DIR = Path(os.path.abspath(''))
if str(PRITHVI2_DIR) not in sys.path:
    sys.path.insert(0, str(PRITHVI2_DIR))

import config
import model as prithvi_model
import data_loader
import cloud_infill
import segment
import pipeline_adapter

print(f'Torch {torch.__version__}  CUDA available: {torch.cuda.is_available()}')

## 1. Configuration & Site Setup

In [ ]:
# ---- Point to a site processed by the pipeline ----
SITE_PATH = Path('/home/walter_littor_al/geotools_sites/Fuvahmulah')
TIFF_DIR  = SITE_PATH / 'TARGETS'
MASK_DIR  = SITE_PATH / 'MASK'

tiff_files = sorted(TIFF_DIR.glob('*.tif'))
print(f'Found {len(tiff_files)} TIF files in {TIFF_DIR}')
for f in tiff_files[:5]:
    print(f'  {f.name}')

## 2. Load & Inspect a GeoTIFF

In [ ]:
sample_tif = str(tiff_files[0])
arr, meta = data_loader.read_tiff(sample_tif)
print(f'Shape: {arr.shape}  dtype: {arr.dtype}')
for i in range(arr.shape[0]):
    print(f'  Band {i}: min={arr[i].min():.0f}  max={arr[i].max():.0f}  mean={arr[i].mean():.0f}')

# RGB composite  (B04=Red, B03=Green, B02=Blue → indices 2,1,0 in 7-band order)
rgb = np.stack([arr[2], arr[1], arr[0]], axis=0).astype(np.float32)
rgb = np.clip(rgb / 3000, 0, 1).transpose(1, 2, 0)
plt.figure(figsize=(8, 8))
plt.imshow(rgb)
plt.title(f'{Path(sample_tif).stem} — True colour')
plt.axis('off')
plt.show()

## 3. Band Selection & Normalisation

In [ ]:
# Select Prithvi-compatible bands from the 7-band TIFF
prithvi_bands = data_loader.select_bands(arr, num_src_bands=arr.shape[0])
print(f'Selected bands shape: {prithvi_bands.shape}')
print(f'Expected band order: {config.HLS_BANDS}')

# Normalise using HLS statistics
normed = data_loader.normalize(prithvi_bands)
print(f'After normalisation: min={normed.min():.2f}  max={normed.max():.2f}')

## 4. Load MAE Model

In [ ]:
mae = prithvi_model.load_mae()
device = prithvi_model.resolve_device()
n_params = sum(p.numel() for p in mae.parameters())
print(f'MAE loaded on {device}  —  {n_params/1e6:.1f}M parameters')

## 5. Test MAE Reconstruction

In [ ]:
# Prepare a single 224×224 tile
tiles, info = data_loader.tile_image(normed, tile_size=config.IMG_SIZE)
print(f'Tiled into {len(tiles)} patches of shape {tiles[0].shape}')

# Forward one tile through MAE with 75% masking
x = torch.tensor(tiles[0], dtype=torch.float32).unsqueeze(0)  # (1, C, H, W)
x = x.unsqueeze(2).to(device)  # (1, C, 1, H, W) — single temporal frame

mae.eval()
with torch.no_grad():
    loss, pred, mask = mae(x, None, None, mask_ratio=0.75)

recon = mae.unpatchify(pred).squeeze().cpu().numpy()  # (C, H, W)
orig  = tiles[0]

# Visualise band 0 (Blue)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(orig[0], cmap='gray'); axes[0].set_title('Original (Blue band)')
axes[1].imshow(recon[0], cmap='gray'); axes[1].set_title('Reconstructed')
axes[2].imshow(np.abs(orig[0] - recon[0]), cmap='hot'); axes[2].set_title('|Error|')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()
print(f'Reconstruction loss: {loss.item():.4f}')

## 6. Cloud Infill

In [ ]:
output_dir = str(SITE_PATH / 'TARGETS' / 'prithvi2_cloudless')
os.makedirs(output_dir, exist_ok=True)

# Infill a single image
result_path = cloud_infill.infill_image(sample_tif, output_dir, mae_model=mae)
print(f'Infilled: {result_path}')

# Compare original vs infilled
infilled_arr, _ = data_loader.read_tiff(result_path)
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, data, title in zip(axes, [arr, infilled_arr], ['Original', 'Cloud-infilled']):
    rgb_ = np.clip(np.stack([data[2], data[1], data[0]]).astype(np.float32) / 3000, 0, 1).T
    ax.imshow(np.transpose(rgb_, (1, 0, 2)))
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

## 7. Segmentation Inference

In [ ]:
segmenter = segment.Prithvi2Seg(seg_checkpoint=None)  # MAE-features, no fine-tuned head yet

mask_path = segmenter.mask_from_img(sample_tif)
print(f'Mask saved: {mask_path}')

mask_img = np.array(plt.imread(mask_path))
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(rgb); axes[0].set_title('True colour')
axes[1].imshow(mask_img, cmap='gray'); axes[1].set_title('Prithvi2 mask (white=land)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 8. Compare with YOLO Masks

In [ ]:
from PIL import Image

# Find matching YOLO mask
stem = Path(sample_tif).stem
yolo_mask_path = MASK_DIR / f'{stem}.png'
if yolo_mask_path.exists():
    yolo_mask = np.array(Image.open(yolo_mask_path).convert('L')) > 127
    prithvi_mask = np.array(Image.open(mask_path).convert('L')) > 127

    # Resize prithvi mask to YOLO mask size for comparison
    from PIL import Image as PILImage
    prithvi_resized = np.array(
        PILImage.fromarray(prithvi_mask.astype(np.uint8) * 255).resize(
            (yolo_mask.shape[1], yolo_mask.shape[0]), PILImage.NEAREST
        )
    ) > 127

    # IoU
    intersection = (yolo_mask & prithvi_resized).sum()
    union = (yolo_mask | prithvi_resized).sum()
    iou = intersection / union if union > 0 else 0.0

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(yolo_mask, cmap='gray'); axes[0].set_title('YOLO mask')
    axes[1].imshow(prithvi_resized, cmap='gray'); axes[1].set_title('Prithvi2 mask')
    axes[2].imshow(yolo_mask.astype(int) - prithvi_resized.astype(int), cmap='RdBu')
    axes[2].set_title(f'Difference  (IoU={iou:.3f})')
    for ax in axes: ax.axis('off')
    plt.tight_layout(); plt.show()
else:
    print(f'No YOLO mask found at {yolo_mask_path}')

## 9. Batch Site Processing

In [ ]:
p2 = pipeline_adapter.Prithvi2Pipeline()

# Cloud infill all TIFFs
cloudless_paths = p2.cloud_infill_folder(str(TIFF_DIR), output_dir)
print(f'Cloud-infilled {len(cloudless_paths)} images')

# Segment all
mask_paths = p2.mask_from_folder(output_dir)
print(f'Generated {len(mask_paths)} segmentation masks')